# MarkerRepo

## 0. Setup & Imports

In [ ]:
import os
import markerrepo.marker_repo as mr
import markerrepo.wrappers as wrap
import markerrepo.annotation as annot
import markerrepo.parsing as pars
import markerrepo.homology as homol
import markerrepo.scoring as score
import markerrepo.utils as utils
import markerrepo.plotting as plot
import markerrepo.validate_yaml as validate
import scanpy as sc

%load_ext autoreload
%autoreload 2

In [ ]:
repo_path = os.path.abspath("../")

## 1. Search & Combine Marker Lists

This section explains how to search the marker repository to further process and use a selection of lists. This happens mainly by selecting, combining, formatting and finally exporting the selection made. Wrapper functions facilitate this process and perform all four steps sequentially after a function call.

### Guided Search

By using the <b>Guided Search</b>, a selection of marker lists can be compiled to suit the individual needs. First, a column of the available metadata of all lists is selected to search in it afterwards. This process can be repeated as often as you like until you have exactly the lists you need in the selection. Finally, the selection can be returned as metadata or as a finished marker list in the form of a DataFrame.

Return search results as metadata DataFrame.

In [ ]:
results = mr.guided_search(repo_path=repo_path, out="metadata")
display(results)

Return search results as marker list DataFrame.

In [ ]:
results = mr.guided_search(repo_path=repo_path, out="marker_list")
display(results)

Export the combined marker list using Gene Symbols as identifiers.

In [ ]:
mr.export_marker_list(results, file_name="Combined", marker_id="symbol")

Export the combined marker list using Ensembl IDs as identifiers.

In [ ]:
mr.export_marker_list(results, file_name="Combined", marker_id="ensembl")

### Wrapper Functions

<b>Use the guided search to create a new marker list in "two column" style.</b><br>
<br>This format can be used for custom cell type annotations via SCSA, for example. The function combines, formats and exports the new marker list accordingly. It returns the path of the created file. You can adjust the file name by using the "file_name" parameter.

Get Gene Symbols

In [ ]:
wrap.convert_markers(repo_path=repo_path, gs=True, style="two_column", file_name="Gene_symbols")

Get Ensembl IDs

In [ ]:
wrap.convert_markers(repo_path=repo_path, gs=True, style="two_column", file_name="Ensembl_IDs", ensembl=True)

<b>Use the guided search to create a new marker list in "score" style.</b><br>
<br>
When you choose this function, the output is formatted into a three-column layout: the first column for the marker name, the second for the cell type it's associated with, and the third for the weight or score assigned to each marker.

In [ ]:
wrap.convert_markers(repo_path=repo_path, gs=True, style="score", path=".", file_name="mouse_panglao", organism="Mm")

## 2. Cell Type Annotation

In this section, clustered h5ad files can be annotated using the MarkerRepo or SCSA.

### Settings

Specify the h5ad file which is going to be annotated.

In [ ]:
h5ad_path = "./test_data/adata_annotation.h5ad"

Load anndata and list all possible settings.

In [ ]:
adata = sc.read_h5ad(h5ad_path)
annot.list_possible_settings(repo_path, adata=adata)

In [ ]:
adata.var

Enter general annotation settings.

In [ ]:
# Taxonomy ID or Organism Name
# e.g., "human" or 9606
organism = None

# Column in .obs table where ranked genes groups are stored
# e.g., "rank_genes_groups"
# Enter None if no ranking has been performed yet
rank_genes_column = None

# Column in .var table where gene symbols or Ensembl IDs are stored
# Enter None if the index column of the .var table already has gene symbols or Ensembl IDs
# that you want to use for your annotation
genes_column = None

# The .obs table column of the clustering you want to annotate (e.g., "leiden" or "louvain")
# If None, you can pick one interactively
clustering_column = None

# Whether to use "Genes" or "Genomic regions"
# If None, select interactively
marker_type = None

# RNA or ATAC
# If None, select interactively
omic = None

# Specify whether your index of the .var tables are Ensembl IDs (True) or gene symbols (False)
ensembl = mr.check_ensembl(adata)

# Name of the column to add with the final cell type annotation
# If None, all annotation columns will be kept
celltype_column_name = None

# Whether to delete the created marker lists after annotation or not
delete_lists = False

Specify Marker Lists for Annotation using column specific terms:

- `key`: Specify the column to search in. Use `None` to search across all columns.
  - Example columns include `Source`, `Organism name`, etc.
- `value`: Define your search terms. Use `-` to exclude keywords and `+` to ensure the keyword must be present.
  - Separate multiple keywords with a comma. For example: `["+panglao.se", "+mouse"]` to include lists from 'panglao.se' and related to 'mouse'.

In [ ]:
column_specific_terms = {"Organism name":"human"}

Adjust various settings for marker lists, like 'style' or 'file_name', otherwise the default settings will be used. A dictionary corresponds to a marker list.

Example:
```python
settings = [
    {
        "style": "two_column",
        "file_name": "basic_markers"
    },
    {
        "style": "score",
        "column_specific_terms": {
            "Source": "panglao",
            "Tissue": "heart"
        },
        "file_name": "heart_panglao"
    },
    {
        "force_homology": True,
        "file_name": "homology_markers"
    }
]
```

In [ ]:
cml_parameters = [{"style":"two_column", "file_name":"two_column"},
            {"style":"score", "file_name":"score"}]

Validate general annotation settings and the mr_parameters.

In [ ]:
wrap.validate_settings(
    cml_parameters=cml_parameters,
    repo_path=repo_path,
    adata=adata,
    organism=organism,
    rank_genes_column=rank_genes_column,
    genes_column=genes_column,
    clustering_column=clustering_column,
    ensembl=ensembl,
    column_specific_terms=column_specific_terms
)

### Prepare adata

Set genes to index, if not already done.

In [ ]:
if genes_column:
    adata.var.reset_index(inplace=True)  # remove old index values and save them in the column ['index']
    adata.var.set_index(genes_column, inplace=True)  # set genes as index
    adata.var.index = adata.var.index.astype('str')  # to avoid index being categorical
    adata.var_names_make_unique(join='_')

    # update ensembl if gene identifier has changed
    ensembl = mr.check_ensembl(adata)

display(adata.var)

### Create suitable marker list(s)

<details>
    <summary>Click here to see/collapse the function description</summary>
    <p><b>Function Call:</b> create_multiple_marker_lists</p>
    <p>This function calls 'create_marker_lists' with multiple parameter sets to create marker lists. It iterates over each dictionary within a list, using its contents to call 'create_marker_lists'. Default values are assigned for any parameters missing from a dictionary, but these can be overridden by individual dictionary entries.</p>
    <p><b>Parameters (excerpt):</b></p>
    <ul>
        <li><b>settings:</b> list of dict, default [{}] - A list of dictionaries where each dictionary contains parameters for a single call to 'create_marker_lists'.</li>
        <li><b>style:</b> str, default "score" - Determines the style of the marker lists. Available options include "two_column", "score", "ui", and "panglao".</li>
        <li><b>force_homology:</b> bool, default False - If set to True, the function will attempt to create marker lists via homology, even if marker lists for the given organism already exist.</li>
        <li><b>show_lists:</b> bool, default True - If True, the function displays the marker lists of the query post-creation.</li>
        <li><b>column_specific_terms:</b> dict, default None - A dictionary with column names as keys and lists of search terms as values.</li>
        <li><b>adata:</b> AnnData, default None - If provided, the function adds the marker list IDs to the .uns table of the AnnData object.</li>
    </ul>
    <p><b>Returns:</b></p>
    <ul>
        <li><b>list of str:</b> A list of all paths to the created marker lists.</li>
    </ul>
</details>

The paths of the marker lists will be stored in the <b>marker_lists</b> variable. They will work as input for the actual cell type annotation of the next cell.

In [ ]:
marker_lists = wrap.create_multiple_marker_lists(
    cml_parameters=cml_parameters,
    repo_path=repo_path,
    organism=organism,
    ensembl=ensembl,
    column_specific_terms=column_specific_terms,
    show_lists=True,
    adata=adata,
    suffix="tags.tissue",
    marker_type=marker_type
)

### Annotate adata using the created list(s)

<details>
    <summary>Click here to see/collapse the function description</summary>
    <p><b>Function Call:</b> run_annotation</p>
    <p>This function performs annotations on single cell data and allows the user to choose between different annotation methods. It supports both MarkerRepo and SCSA annotations.</p>
    <p><b>Parameters (excerpt):</b></p>
    <ul>
        <li><b>adata:</b> AnnData - The AnnData object to annotate.</li>
        <li><b>SCSA:</b> bool, default True - Specifies whether to use SCSA annotation.</li>
        <li><b>marker_lists:</b> list of str, default [] - Paths to marker list files for annotation.</li>
        <li><b>rank_genes_column:</b> str, default None - The column in .uns containing rank genes scores.</li>
        <li><b>clustering_column:</b> str, default None - The column in .obs containing clustering information.</li>
        <li><b>show_ct_tables:</b> bool, default False - If True, displays tables of the MarkerRepo annotation results.</li>
        <li><b>show_plots:</b> bool, default False - If True, displays UMAP plots related to the annotation.</li>
        <li><b>show_comparison:</b> bool, default False - If True, shows a comparison table of the different annotations.</li>
        <li><b>celltype_column_name:</b> str, default None - Names the selected cell type annotation column.</li>
    </ul>
</details>

In [ ]:
wrap.run_annotation(
    adata,
    SCSA=False,
    marker_lists=marker_lists,
    reference_obs=None,
    show_comparison=True,
    clustering_column=clustering_column,
    rank_genes_column=rank_genes_column,
    ignore_overwrite=False,
    verbose=False,
    show_plots=True,
    show_ct_tables=True,
    celltype_column_name=celltype_column_name,
    omic=omic,
    upstream_offset=1000,
    downstream_offset=1000
)

Show new annotation column.

In [ ]:
adata.obs

In [ ]:
adata.uns["MarkerRepo"]

Delete created marker lists.

In [ ]:
if delete_lists:
    mr.delete_files(marker_lists)

## 3. Create Marker Lists (YAML)

This section facilitates the creation of YAML marker lists.

**Components of a YAML Marker List**
- **Metadata**: Entered manually, this part includes essential details about the marker list.
- **Markers**: Derived from a tab-delimited file with one or two columns.

**Process Workflow**
1. **Settings**: Specify the path to the marker file and details about its columns.
2. **Create yaml list**: Essential information is entered (name, organism, marker type), markers are filtered and expanded using whitelists, and the list is saved as a YAML file.
3. **Validation**: Perform a validation of the marker list.

### Blood cell type marker list

The path of the list to be added to the marker repo.<br>
The file must consist of one marker per line or two tab separated columns (marker and info like cell type).

In [ ]:
list_path = "./test_data/marker_list_blood.tsv"

Additional information about the columns.<br>
`marker_col`: The column where the markers are stored<br>
`info_col`: The column where the cell type information is stored

In [ ]:
marker_col = 0
info_col = 1

Enter essential information. If None, select interactively when executing <b>create_yaml_list</b>.

In [ ]:
# The name of the marker list (e.g. "zebrafish_heart")
list_name = None
# Organism name (e.g. "human") or taxon ID (e.g. 9606)
organism = None
# "Genes" or "Genomic regions"
marker_type = None

In [ ]:
yaml_path = wrap.create_yaml_list(
    list_path,
    list_name=list_name,
    organism=organism,
    marker_type=marker_type,
    marker_col=marker_col,
    info_col=info_col,
    output_path=".",
    repo_path=repo_path)

### Validation

Check whether the format of the yaml file is correct.

In [ ]:
if validate.validate_file(utils.read_in_yaml(f"{yaml_path}", marker_list=False), repo_path=repo_path):
    print(f"YAML file is valid.")

### Brain cell type marker list

The path of the list to be added to the marker repo.<br>
The file must consist of one marker per line or two tab separated columns (marker and info like cell type).

In [ ]:
list_path = "./test_data/marker_list_brain.tsv"

Enter essential information. If None, select interactively when executing <b>create_yaml_list</b>.

In [ ]:
# The name of the marker list (e.g. "zebrafish_heart")
list_name = None
# Organism name (e.g. "human") or taxon ID (e.g. 9606)
organism = None
# "Genes" or "Genomic regions"
marker_type = None

In [ ]:
yaml_path = wrap.create_yaml_list(
    list_path,
    list_name=list_name,
    organism=organism,
    marker_type=marker_type,
    marker_col=marker_col,
    info_col=info_col,
    output_path=".",
    repo_path=repo_path)

### Validation

Check whether the format of the yaml file is correct.

In [ ]:
if validate.validate_file(utils.read_in_yaml(f"{yaml_path}", marker_list=False), repo_path=repo_path):
    print(f"YAML file is valid.")